In [5]:
import pandas as pd


In [7]:
data = pd.read_csv("hotel_bookings.csv")

In [9]:
data.head(10)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03
6,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,No Deposit,NaN,NaN,0,Transient,107.0,0,0,Check-Out,2015-07-03
7,Resort Hotel,0,9,2015,July,27,1,0,2,2,...,No Deposit,303.0,NaN,0,Transient,103.0,0,1,Check-Out,2015-07-03
8,Resort Hotel,1,85,2015,July,27,1,0,3,2,...,No Deposit,240.0,NaN,0,Transient,82.0,0,1,Canceled,2015-05-06
9,Resort Hotel,1,75,2015,July,27,1,0,3,2,...,No Deposit,15.0,NaN,0,Transient,105.5,0,0,Canceled,2015-04-22


In [11]:
# Shape before
print("Before deduplication:", data.shape)

Before deduplication: (119390, 32)


In [15]:
# Drop duplicate rows
data.drop_duplicates(inplace=True)

# Shape after
print("After deduplication:", data.shape)


After deduplication: (87396, 32)


In [17]:
# Overview of nulls
null_counts = data.isnull().sum()
print(null_counts[null_counts > 0].sort_values(ascending=False))


company     82137
agent       12193
country       452
children        4
dtype: int64


In [21]:
# Fill missing children with 0
data['children'].fillna(0, inplace=True)

# Fill agent and company with 0 (or 'Unknown')
data['agent'].fillna(0, inplace=True)
data['company'].fillna(0, inplace=True)

# Fill missing country with 'Unknown'
data['country'].fillna('Unknown', inplace=True)


In [23]:
# Overview of nulls
null_counts = data.isnull().sum()
print(null_counts[null_counts > 0].sort_values(ascending=False))

Series([], dtype: int64)


In [27]:
# Ensure agent and company are integers (after filling NaNs)
data['agent'] = data['agent'].astype(int)
data['company'] = data['company'].astype(int)
data['children'] = data['children'].astype(int)


In [29]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 87396 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87396 non-null  object 
 1   is_canceled                     87396 non-null  int64  
 2   lead_time                       87396 non-null  int64  
 3   arrival_date_year               87396 non-null  int64  
 4   arrival_date_month              87396 non-null  object 
 5   arrival_date_week_number        87396 non-null  int64  
 6   arrival_date_day_of_month       87396 non-null  int64  
 7   stays_in_weekend_nights         87396 non-null  int64  
 8   stays_in_week_nights            87396 non-null  int64  
 9   adults                          87396 non-null  int64  
 10  children                        87396 non-null  int32  
 11  babies                          87396 non-null  int64  
 12  meal                            8739

In [33]:
# Total stay nights
data['total_nights'] = data['stays_in_weekend_nights'] + data['stays_in_week_nights']

# Total revenue (only for non-canceled bookings)
data['revenue'] = data.apply(lambda row: row['adr'] * row['total_nights'] if row['is_canceled'] == 0 else 0, axis=1)

# Create unified arrival_date
data['arrival_date'] = pd.to_datetime(
    data['arrival_date_year'].astype(str) + '-' +
    data['arrival_date_month'] + '-' +
    data['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)


In [35]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 87396 entries, 0 to 119389
Data columns (total 35 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           87396 non-null  object        
 1   is_canceled                     87396 non-null  int64         
 2   lead_time                       87396 non-null  int64         
 3   arrival_date_year               87396 non-null  int64         
 4   arrival_date_month              87396 non-null  object        
 5   arrival_date_week_number        87396 non-null  int64         
 6   arrival_date_day_of_month       87396 non-null  int64         
 7   stays_in_weekend_nights         87396 non-null  int64         
 8   stays_in_week_nights            87396 non-null  int64         
 9   adults                          87396 non-null  int64         
 10  children                        87396 non-null  int32         
 11  babies

In [37]:
data.to_csv("cleaned_hotel_bookings.csv", index=False)


In [71]:
data.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date', 'total_nights',
       'revenue', 'arrival_date'],
      dtype='object')

In [43]:
data['hotel'].unique()

array(['Resort Hotel', 'City Hotel'], dtype=object)

In [73]:
import pandas as pd

def generate_dim_date(arrival_dates, reservation_dates):
    """
    Generate a dim_date table from arrival and reservation date columns.

    Parameters:
    - arrival_dates: pd.Series of datetime objects
    - reservation_dates: pd.Series of datetime objects

    Returns:
    - dim_date: pd.DataFrame with full calendar attributes and surrogate keys
    """

    # Combine both date columns and remove duplicates
    all_dates = pd.concat([arrival_dates, reservation_dates]).dropna().drop_duplicates().reset_index(drop=True)
    
    # Sort dates
    all_dates = pd.to_datetime(all_dates).sort_values().reset_index(drop=True)
    
    # Create the dim_date DataFrame
    dim_date = pd.DataFrame({'date': all_dates})
    dim_date['date_id'] = dim_date.index + 1  # Surrogate key

    # Add derived date parts
    dim_date['year'] = dim_date['date'].dt.year
    dim_date['month'] = dim_date['date'].dt.month
    dim_date['month_name'] = dim_date['date'].dt.strftime('%B')
    dim_date['day'] = dim_date['date'].dt.day
    dim_date['weekday_num'] = dim_date['date'].dt.weekday + 1  # Monday = 1
    dim_date['weekday_name'] = dim_date['date'].dt.strftime('%A')
    dim_date['week_number'] = dim_date['date'].dt.isocalendar().week

    return dim_date


In [75]:
dim_date = generate_dim_date(data['arrival_date'], data['reservation_status_date'])

print(dim_date.head())

        date  date_id  year  month month_name  day  weekday_num weekday_name  \
0 2014-10-17        1  2014     10    October   17            5       Friday   
1 2014-11-18        2  2014     11   November   18            2      Tuesday   
2 2015-01-01        3  2015      1    January    1            4     Thursday   
3 2015-01-02        4  2015      1    January    2            5       Friday   
4 2015-01-18        5  2015      1    January   18            7       Sunday   

   week_number  
0           42  
1           47  
2            1  
3            1  
4            3  


In [77]:
dim_date.to_csv('dim_date.csv', index=False)

In [79]:
def create_dimension(df, column_name, id_name):
    unique_values = df[column_name].dropna().unique()
    dim_df = pd.DataFrame({column_name: sorted(unique_values)})
    dim_df[id_name] = range(1, len(dim_df) + 1)
    return dim_df


In [83]:
dim_hotel = create_dimension(data, 'hotel', 'hotel_id')
dim_meal = create_dimension(data, 'meal', 'meal_id')
dim_country = create_dimension(data, 'country', 'country_id')
dim_market_segment = create_dimension(data, 'market_segment', 'market_segment_id')
dim_distribution_channel = create_dimension(data, 'distribution_channel', 'distribution_channel_id')
dim_customer_type = create_dimension(data, 'customer_type', 'customer_type_id')
dim_deposit_type = create_dimension(data, 'deposit_type', 'deposit_type_id')
dim_reservation_status = create_dimension(data, 'reservation_status', 'reservation_status_id')


In [87]:
all_room_types = pd.concat([
    data['reserved_room_type'],
    data['assigned_room_type']
]).dropna().unique()

dim_room_type = pd.DataFrame({
    'room_type': sorted(all_room_types),
    'room_type_id': range(1, len(all_room_types) + 1)
})

print(dim_room_type)

   room_type  room_type_id
0          A             1
1          B             2
2          C             3
3          D             4
4          E             5
5          F             6
6          G             7
7          H             8
8          I             9
9          K            10
10         L            11
11         P            12


In [93]:
data['reservation_status_date'] = pd.to_datetime(data['reservation_status_date'])

In [95]:
data = data.merge(dim_hotel, on='hotel', how='left')
data = data.merge(dim_meal, on='meal', how='left')
data = data.merge(dim_country, on='country', how='left')
data = data.merge(dim_market_segment, on='market_segment', how='left')
data = data.merge(dim_distribution_channel, on='distribution_channel', how='left')
data = data.merge(dim_customer_type, on='customer_type', how='left')
data = data.merge(dim_deposit_type, on='deposit_type', how='left')
data = data.merge(dim_reservation_status, on='reservation_status', how='left')

data = data.merge(
    dim_date[['date', 'date_id']].rename(columns={'date': 'arrival_date', 'date_id': 'arrival_date_id'}), 
    on='arrival_date', 
    how='left'
)
data = data.merge(
    dim_date[['date', 'date_id']].rename(columns={'date': 'reservation_status_date', 'date_id': 'reservation_status_date_id'}), 
    on='reservation_status_date', 
    how='left'
)


data = data.merge(
    dim_room_type.rename(columns={'room_type': 'reserved_room_type', 'room_type_id': 'reserved_room_type_id'}),
    on='reserved_room_type',
    how='left'
)
data = data.merge(
    dim_room_type.rename(columns={'room_type': 'assigned_room_type', 'room_type_id': 'assigned_room_type_id'}),
    on='assigned_room_type',
    how='left'
)

In [99]:
print(data.columns.tolist())


['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date', 'total_nights', 'revenue', 'arrival_date', 'hotel_id_x', 'meal_id_x', 'country_id_x', 'market_segment_id_x', 'distribution_channel_id_x', 'customer_type_id_x', 'deposit_type_id_x', 'reservation_status_id_x', 'arrival_date_id_x', 'hotel_id_y', 'meal_id_y', 'country_id_y', 'market_segment_id_y', 'distribution_channel_id_y', 'customer_type_id_y', 'deposit_type_id_y', 'reservation_s

In [103]:
data = data.rename(columns={
    'hotel_id_x': 'hotel_id',
    'meal_id_x': 'meal_id',
    'country_id_x': 'country_id',
    'market_segment_id_x': 'market_segment_id',
    'distribution_channel_id_x': 'distribution_channel_id',
    'customer_type_id_x': 'customer_type_id',
    'deposit_type_id_x': 'deposit_type_id',
    'reservation_status_id_x': 'reservation_status_id',
    'arrival_date_id_x': 'arrival_date_id',
})


In [105]:
data = data.drop(columns=[
    'hotel_id_y', 'meal_id_y', 'country_id_y',
    'market_segment_id_y', 'distribution_channel_id_y',
    'customer_type_id_y', 'deposit_type_id_y',
    'reservation_status_id_y', 'arrival_date_id_y'
])


In [107]:
fact_bookings = data[[
    'hotel_id',
    'arrival_date_id',
    'reservation_status_date_id',
    'meal_id',
    'country_id',
    'market_segment_id',
    'distribution_channel_id',
    'customer_type_id',
    'deposit_type_id',
    'reservation_status_id',
    'reserved_room_type_id',
    'assigned_room_type_id',
    
    'is_canceled',
    'lead_time',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'is_repeated_guest',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'booking_changes',
    'agent',
    'company',
    'days_in_waiting_list',
    'adr',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'total_nights',
    'revenue'
]]


In [109]:
fact_bookings.insert(0, 'booking_id', range(1, len(fact_bookings) + 1))


In [111]:
fact_bookings.to_csv('fact_bookings.csv', index=False)


In [113]:
dim_hotel.to_csv('dim_hotel.csv', index=False)
dim_meal.to_csv('dim_meal.csv', index=False)
dim_country.to_csv('dim_country.csv', index=False)
dim_market_segment.to_csv('dim_market_segment.csv', index=False)
dim_distribution_channel.to_csv('dim_distribution_channel.csv', index=False)
dim_customer_type.to_csv('dim_customer_type.csv', index=False)
dim_deposit_type.to_csv('dim_deposit_type.csv', index=False)
dim_reservation_status.to_csv('dim_reservation_status.csv', index=False)
dim_room_type.to_csv('dim_room_type.csv', index=False)


In [117]:
pip install mysql-connector-python



   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB 165.2 kB/s eta 0:01:39
   ---------------------------------------- 0.0/16.4 MB 219.4 kB/s eta 0:01:15
   ---------------------------------------- 0.1/16.4 MB 219.0 kB/s eta 0:01:15
   ---------------------------------------- 0.1/16.4 MB 393.8 kB/s eta 0:00:42
    --------------------------------------- 0.2/16.4 MB 850.1 kB/s eta 0:00:19
   - -------------------------------------- 0.6/16.4 MB 1.8 MB/s eta 0:00:09
   -- ------------------------------------- 0.9/16.4 MB 2.2 MB/s eta 0:00:08
   -- ------------------------------------- 1.2/16.4 MB 2.9 MB/s eta 0:00:06
   --- ------------------------------------ 1.3/16.4 MB 2.7 MB/s eta 0:00:06
   --- ------------------------------------ 1.6/16.4 MB 3.0 MB/s eta 0:00:05
   ---- -

In [127]:
import mysql.connector
import pandas as pd

# --- MySQL Configuration ---
import os

config = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME', 'hotel_bookings')
}

if not config['user'] or not config['password']:
    raise ValueError("Please configure DB_USER and DB_PASSWORD.")

# --- CSV File Path ---
csv_file = '../data/fact_bookings.csv'

# --- Load CSV in chunks ---
chunksize = 10000
data_iter = pd.read_csv(csv_file, chunksize=chunksize)

# --- Connect to MySQL ---
conn = mysql.connector.connect(**config)
cursor = conn.cursor()

# --- Prepare INSERT query for 31 columns ---
insert_query = """
INSERT INTO fact_bookings (
    hotel_id,
    arrival_date_id,
    reservation_status_date_id,
    meal_id,
    country_id,
    market_segment_id,
    distribution_channel_id,
    customer_type_id,
    deposit_type_id,
    reservation_status_id,
    reserved_room_type_id,
    assigned_room_type_id,
    is_canceled,
    lead_time,
    stays_in_weekend_nights,
    stays_in_week_nights,
    adults,
    children,
    babies,
    is_repeated_guest,
    previous_cancellations,
    previous_bookings_not_canceled,
    booking_changes,
    agent,
    company,
    days_in_waiting_list,
    adr,
    required_car_parking_spaces,
    total_of_special_requests,
    total_nights,
    revenue
)
VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s
)
"""

# --- Specify only the needed columns in correct order ---
columns = [
    'hotel_id',
    'arrival_date_id',
    'reservation_status_date_id',
    'meal_id',
    'country_id',
    'market_segment_id',
    'distribution_channel_id',
    'customer_type_id',
    'deposit_type_id',
    'reservation_status_id',
    'reserved_room_type_id',
    'assigned_room_type_id',
    'is_canceled',
    'lead_time',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'is_repeated_guest',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'booking_changes',
    'agent',
    'company',
    'days_in_waiting_list',
    'adr',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'total_nights',
    'revenue'
]

# --- Insert each chunk ---
for chunk in data_iter:
    chunk = chunk.where(pd.notnull(chunk), None)  # Replace NaNs with None
    chunk = chunk[columns]  # Keep only necessary columns
    values = [tuple(row) for row in chunk.values]
    cursor.executemany(insert_query, values)
    conn.commit()
    print(f"Inserted {len(values)} rows...")

# --- Close connections ---
cursor.close()
conn.close()
print("Import completed successfully!")


Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10000 rows...
Inserted 10

In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

try:
    import pandas
    print("Pandas version:", pandas.__version__)
except ImportError:
    print("Pandas: NOT INSTALLED")

try:
    import mysql.connector
    print("MySQL Connector: INSTALLED")
except ImportError:
    print("MySQL Connector: NOT INSTALLED")
    

Python executable: C:\Users\hp\anaconda3\python.exe
Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
Pandas version: 2.3.3
MySQL Connector: INSTALLED
